
# 03 — Joint Classifier

## 1. Experimental Objective

The objective of this experiment is to establish a joint-classification
baseline for evaluating visual compositional generalization.

The model treats each valid shape–color combination as an independent
class rather than predicting shape and color separately.

The experiment investigates how this classification strategy performs
on familiar combinations encountered during training and previously
unseen combinations during evaluation.

The results will provide a baseline for comparison with the factorized
classification approach implemented in the subsequent experiment.

### Research Question

How well can a conventional joint classifier recognize object
combinations that were not observed during training?

### Experimental Hypothesis

The joint classifier is expected to perform better on familiar
shape–color combinations than on unseen combinations, since each
combination is represented as an independent class.



## 2. Experimental Setup

### 2.1 Computational Environment

The experiments are conducted using the Kaggle notebook environment.

The computational environment is inspected before model development
to identify the available hardware resources and software versions.

### 2.2 Deep Learning Framework

PyTorch is used as the primary deep learning framework for implementing
and training the joint classification model.

### 2.3 Reproducibility

The experimental environment, library versions, and available
computational resources are recorded to support reproducibility.

A fixed random seed will be used in the subsequent implementation
to improve consistency across experimental runs.


In [1]:

import sys
import torch
import torchvision
import numpy as np
import pandas as pd
import sklearn

# Computational environment
print("=" * 50)
print("EXPERIMENTAL ENVIRONMENT")
print("=" * 50)

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

# Hardware information
print("\n" + "=" * 50)
print("COMPUTATIONAL RESOURCES")
print("=" * 50)

print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("Running on CPU")

# Random seed
RANDOM_SEED = 42

print(f"\nRandom seed: {RANDOM_SEED}")


EXPERIMENTAL ENVIRONMENT
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130
NumPy version: 2.1.3
Pandas version: 2.3.3
Scikit-learn version: 1.6.1

COMPUTATIONAL RESOURCES
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Random seed: 42



### 2.3 Hardware Configuration

The initial environment inspection indicated that CUDA was unavailable
and the installed PyTorch build was CPU-only.

GPU acceleration must be configured before conducting the model
training experiments to ensure efficient execution.

The computational environment will be re-inspected after enabling
the GPU accelerator.



### 2.4 Verified Computational Environment

GPU acceleration was successfully enabled in the Kaggle environment.

The environment provides two NVIDIA Tesla T4 GPUs, with CUDA
available through the installed PyTorch build.

The verified software configuration is:

- Python: 3.13.16
- PyTorch: 2.11.0+cu130
- Torchvision: 0.26.0+cu130
- NumPy: 2.1.3
- Pandas: 2.3.3
- Scikit-learn: 1.6.1

A fixed random seed of 42 is designated for reproducibility.



## 3. Dataset Loading

The CLEVR-CoGenT processed dataset is used for training and evaluating
the joint classification model.

The dataset contains preprocessed object crops organized into training
and evaluation splits.

Before loading the images, the directory structure and metadata files
are inspected to identify image paths, object attributes, and split
assignments.

This inspection ensures that the data-loading pipeline is consistent
with the actual dataset organization.


### 3.1 Inspecting the Dataset Directory

The dataset is mounted under Kaggle's input directory. The directory structure is inspected recursively to locate the image splits, metadata files, and other dataset components. This step identifies the exact paths required for loading the training and evaluation data.

In [15]:
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/maram05ai/clevr-cogent-processed")

for path in sorted(DATASET_ROOT.iterdir()):
    print(f"{path.name} | Directory: {path.is_dir()}")

README.md | Directory: False
trainA | Directory: True
valA | Directory: True
valB_unseen | Directory: True


### 3.2 Inspecting Data Splits

The dataset contains three predefined splits: trainA, valA, and valB_unseen. Their directory structures are inspected to identify how image files and metadata are organized. This inspection is necessary to construct a consistent data-loading pipeline while preserving the predefined experimental splits.

In [16]:
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/maram05ai/clevr-cogent-processed"
)

for split_name in ["trainA", "valA", "valB_unseen"]:
    split_path = DATASET_ROOT / split_name

    print(f"\n{'=' * 50}")
    print(f"Split: {split_name}")
    print(f"{'=' * 50}")

    for item in sorted(split_path.iterdir()):
        print(f"{item.name} | Directory: {item.is_dir()}")


Split: trainA
images | Directory: True
metadata_parts | Directory: True

Split: valA
images | Directory: True
metadata_parts | Directory: True

Split: valB_unseen
images | Directory: True
metadata_parts | Directory: True


### 3.3 Inspecting Images and Metadata Files

Each data split contains an `images` directory and a `metadata_parts` directory. A small sample of their contents is inspected to identify image formats, metadata file formats, and naming conventions. The dataset README is also examined to understand the intended metadata structure before implementing the data-loading pipeline.

In [17]:
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/maram05ai/clevr-cogent-processed"
)

# Inspect the README
readme_path = DATASET_ROOT / "README.md"

print("DATASET README")
print("=" * 60)

if readme_path.exists():
    print(readme_path.read_text()[:6000])
else:
    print("README.md not found.")

# Inspect a small sample from each split
for split_name in ["trainA", "valA", "valB_unseen"]:

    split_path = DATASET_ROOT / split_name

    print(f"\n{'=' * 60}")
    print(f"SPLIT: {split_name}")
    print("=" * 60)

    for folder_name in ["images", "metadata_parts"]:

        folder_path = split_path / folder_name

        print(f"\n{folder_name}:")

        samples = sorted(folder_path.iterdir())[:5]

        for item in samples:
            print(f"- {item.name} | File: {item.is_file()}")

DATASET README
# CLEVR-CoGenT Processed Object Crops

## Overview

A processed object-crop dataset derived from CLEVR-CoGenT
for investigating visual generalization to unseen
attribute-object combinations.

All object crops are RGB JPEG images resized to 112x112 pixels.

## Dataset Splits

| Split | Scenes | Object Crops |
|---|---:|---:|
| trainA | 70,000 | 455,024 |
| valA | 15,000 | 97,900 |
| valB_unseen | 15,000 | 64,830 |
| **Total** | **100,000** | **617,754** |

## Directory Structure

```text
clevr-cogent-processed/
├── trainA/
│   ├── images/
│   └── metadata_parts/
├── valA/
│   ├── images/
│   └── metadata_parts/
├── valB_unseen/
│   ├── images/
│   └── metadata_parts/
├── dataset-metadata.json
└── README.md


SPLIT: trainA

images:
- 000000_00.jpg | File: True
- 000000_01.jpg | File: True
- 000000_02.jpg | File: True
- 000001_00.jpg | File: True
- 000001_01.jpg | File: True

metadata_parts:
- metadata_000000_005000.csv | File: True
- metadata_005000_010000.csv | File: True

### 3.4 Inspecting Metadata Schema

The metadata is stored in multiple CSV files within each split. The first metadata file from each split is inspected to identify the available columns, data types, and sample records.

This step establishes how image filenames are associated with their corresponding object attributes and labels. The metadata schema will guide the implementation of the joint classification dataset.

In [18]:
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/maram05ai/clevr-cogent-processed"
)

for split_name in ["trainA", "valA", "valB_unseen"]:

    metadata_dir = DATASET_ROOT / split_name / "metadata_parts"
    metadata_files = sorted(metadata_dir.glob("*.csv"))

    print("\n" + "=" * 70)
    print(f"SPLIT: {split_name}")
    print("=" * 70)

    if not metadata_files:
        print("No metadata CSV files found.")
        continue

    metadata_path = metadata_files[0]
    df_sample = pd.read_csv(metadata_path, nrows=5)

    print(f"\nMetadata file: {metadata_path.name}")
    print(f"Columns: {df_sample.columns.tolist()}")

    print("\nData types:")
    print(df_sample.dtypes)

    print("\nSample records:")
    display(df_sample)


SPLIT: trainA

Metadata file: metadata_000000_005000.csv
Columns: ['crop_filename', 'source_image', 'scene_index', 'object_index', 'shape', 'color', 'material', 'size', 'pixel_x', 'pixel_y']

Data types:
crop_filename    object
source_image     object
scene_index       int64
object_index      int64
shape            object
color            object
material         object
size             object
pixel_x           int64
pixel_y           int64
dtype: object

Sample records:


,crop_filename,source_image,scene_index,object_index,shape,color,material,size,pixel_x,pixel_y
0,000000_00.jpg,CLEVR_trainA_000000.png,0,0,sphere,yellow,metal,large,131,173
1,000000_01.jpg,CLEVR_trainA_000000.png,0,1,cylinder,cyan,metal,large,252,114
2,000000_02.jpg,CLEVR_trainA_000000.png,0,2,sphere,gray,rubber,small,196,115
3,000001_00.jpg,CLEVR_trainA_000001.png,1,0,cube,blue,metal,large,142,173
4,000001_01.jpg,CLEVR_trainA_000001.png,1,1,sphere,blue,rubber,large,326,134



SPLIT: valA

Metadata file: metadata_000000_005000.csv
Columns: ['crop_filename', 'source_image', 'scene_index', 'object_index', 'shape', 'color', 'material', 'size', 'pixel_x', 'pixel_y']

Data types:
crop_filename    object
source_image     object
scene_index       int64
object_index      int64
shape            object
color            object
material         object
size             object
pixel_x           int64
pixel_y           int64
dtype: object

Sample records:


,crop_filename,source_image,scene_index,object_index,shape,color,material,size,pixel_x,pixel_y
0,000000_00.jpg,CLEVR_valA_000000.png,0,0,cube,gray,rubber,large,323,102
1,000000_01.jpg,CLEVR_valA_000000.png,0,1,cube,brown,rubber,small,244,147
2,000000_02.jpg,CLEVR_valA_000000.png,0,2,sphere,red,rubber,large,375,140
3,000000_03.jpg,CLEVR_valA_000000.png,0,3,sphere,cyan,metal,small,155,111
4,000000_04.jpg,CLEVR_valA_000000.png,0,4,cube,blue,metal,small,205,217



SPLIT: valB_unseen

Metadata file: metadata_000000_005000.csv
Columns: ['crop_filename', 'source_image', 'scene_index', 'object_index', 'shape', 'color', 'material', 'size', 'pixel_x', 'pixel_y']

Data types:
crop_filename    object
source_image     object
scene_index       int64
object_index      int64
shape            object
color            object
material         object
size             object
pixel_x           int64
pixel_y           int64
dtype: object

Sample records:


,crop_filename,source_image,scene_index,object_index,shape,color,material,size,pixel_x,pixel_y
0,000000_02.jpg,CLEVR_valB_000000.png,0,2,cylinder,yellow,rubber,large,115,182
1,000000_03.jpg,CLEVR_valB_000000.png,0,3,cube,purple,rubber,large,203,131
2,000000_04.jpg,CLEVR_valB_000000.png,0,4,cube,red,rubber,large,253,75
3,000001_00.jpg,CLEVR_valB_000001.png,1,0,cube,purple,metal,large,253,74
4,000001_01.jpg,CLEVR_valB_000001.png,1,1,cube,purple,metal,large,321,109
